In [ ]:
import pandas as pd
import sqlite3


In [ ]:
# 1. Загружаем данные
df = pd.read_csv('events.csv')

In [ ]:
# 2. Смотрим базовую информацию
print("Количество строк и колонок:", df.shape)
print("\nПропуски в данных:")
print(df.isnull().sum())

Количество строк и колонок: (5625, 16)

Пропуски в данных:
event_id             0
event_ts             0
event_date           0
user_id              0
session_id           0
event_type           0
product_id        1452
category          1452
device_type          0
traffic_source       0
region               0
order_id          5372
quantity             0
unit_price           0
revenue              0
is_new_user          0
dtype: int64


In [ ]:
# 3. Загружаем данные во временную SQL-базу
conn = sqlite3.connect(':memory:')
df.to_sql('events_clean', conn, index=False)

5625

In [20]:
# 4. SQL-запрос для воронки
query = """
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE WHEN event_type = 'visit' THEN user_id END) AS visitors,
        COUNT(DISTINCT CASE WHEN event_type = 'view_item' THEN user_id END) AS viewers,
        COUNT(DISTINCT CASE WHEN event_type = 'add_to_cart' THEN user_id END) AS cart_adders,
        COUNT(DISTINCT CASE WHEN event_type = 'checkout' THEN user_id END) AS checkouts,
        COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
    FROM events_clean
)
SELECT
    *,
    ROUND(buyers * 100.0 / visitors, 2) AS conversion_rate
FROM funnel;
"""

In [21]:
# 5. Выводим результат SQL-запроса в виде таблицы
result = pd.read_sql(query, conn)
display(result)

,visitors,viewers,cart_adders,checkouts,buyers,conversion_rate
0,520,520,367,260,207,39.81
